# SignTranslator — Antrenare ST-GCN + Transformer (Local)

Model: **Spatial-Temporal Graph Convolutional Network + Transformer** (~10M parametri)
- Pre-antrenare self-supervised pe How2Sign
- Fine-tuning cu clasificare pe limba semnelor romaneasca (1926 glosuri)

## Pasi:
1. Setup + Verificare GPU
2. Configurare cai locale
3. Definire model + dataset + augmentari
4. Pre-antrenare pe How2Sign (self-supervised)
5. Cache keypoints din videourile romanesti
6. Fine-tuning pe date romanesti
7. Verificare model final

## 0. Setup + Verificare GPU

In [11]:
import torch
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponibil: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"Memorie: {mem / 1e9:.1f} GB")

PyTorch: 2.9.0+cu128
CUDA disponibil: True
GPU: Tesla T4
Memorie: 15.6 GB


## 1. Configurare cai

Notebook-ul detecteaza automat daca rulezi **local** sau pe **Colab**.

- **Local (VS Code)**: datele se citesc din directorul proiectului
- **Colab (VS Code remote)**: trebuie sa montezi Google Drive sau sa uploadezi datele

Structura necesara:
```
<ROOT>/
  how2sign_pkls/                    <- pkl-urile How2Sign (~31k fisiere)
  ro_dataset/
    dataset.json
    videos/                          <- ~16k videouri .mp4
  models/                            <- aici se salveaza modelele
  data/ro_cache/                     <- cache keypoints
```

In [12]:
import json

# === DETECTARE AUTOMATA MEDIU ===
IS_COLAB = os.path.exists('/content')

if IS_COLAB:
    print("Detectat: Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')

    # Datele sunt direct in MyDrive/ (nu intr-un subfolder)
    DRIVE_ROOT = '/content/drive/MyDrive'
    HOW2SIGN_PKL_DIR = f'{DRIVE_ROOT}/how2sign_pkls_default_shape/how2sign_pkls_cropTrue_shapeFalse'
    RO_DATASET_JSON = f'{DRIVE_ROOT}/ro-sign-language-recognition/datasets/processed_dataset/dataset.json'
    RO_VIDEOS_DIR = f'{DRIVE_ROOT}/ro-sign-language-recognition/datasets/processed_dataset/videos'
    MODELS_DIR = '/content/models'
    CACHE_DIR = '/content/ro_cache'
else:
    print("Detectat: Rulare locala (VS Code)")
    PROJECT_ROOT = os.getcwd()
    HOW2SIGN_PKL_DIR = os.path.join(PROJECT_ROOT, 'how2sign_pkls_default_shape', 'how2sign_pkls_cropTrue_shapeFalse')
    RO_DATASET_JSON = os.path.join(PROJECT_ROOT, 'ro-sign-language-recognition', 'datasets', 'processed_dataset', 'dataset.json')
    RO_VIDEOS_DIR = os.path.join(PROJECT_ROOT, 'ro-sign-language-recognition', 'datasets', 'processed_dataset', 'videos')
    MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
    CACHE_DIR = os.path.join(PROJECT_ROOT, 'data', 'ro_cache')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# === VERIFICARE DATE ===
missing = []
if not os.path.isdir(HOW2SIGN_PKL_DIR):
    missing.append(f"  LIPSESTE: {HOW2SIGN_PKL_DIR}")
if not os.path.isfile(RO_DATASET_JSON):
    missing.append(f"  LIPSESTE: {RO_DATASET_JSON}")
if not os.path.isdir(RO_VIDEOS_DIR):
    missing.append(f"  LIPSESTE: {RO_VIDEOS_DIR}")

if missing:
    print("\n*** EROARE: Datele nu sunt gasite! ***")
    for m in missing:
        print(m)
    raise FileNotFoundError("Verifica structura de foldere pe Drive.")

# Totul OK
pkl_files = sorted([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
print(f"\nHow2Sign PKL-uri: {len(pkl_files)}")

with open(RO_DATASET_JSON, 'r', encoding='utf-8') as f:
    ro_data = json.load(f)
print(f"Glosuri romanesti: {len(ro_data)}")

ro_videos = [f for f in os.listdir(RO_VIDEOS_DIR) if f.endswith('.mp4')]
print(f"Videouri romanesti: {len(ro_videos)}")
print(f"\nModele se salveaza in: {MODELS_DIR}")
print(f"Cache keypoints in: {CACHE_DIR}")
print("\nTotul OK!")

Detectat: Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

How2Sign PKL-uri: 30997
Glosuri romanesti: 1926
Videouri romanesti: 16141

Modele se salveaza in: /content/models
Cache keypoints in: /content/ro_cache

Totul OK!


## 2. Definire Model — ST-GCN + Transformer

Arhitectura:
```
Input (B, T, 75, 3)
    -> [Graph Conv x4] relatii spatiale schelet
    -> [Multi-Scale TCN] dinamica temporala
    -> [Transformer x4] atentie globala
    -> [Classifier] 1926 clase
```

In [13]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional


def build_skeleton_adjacency(num_joints: int = 75) -> torch.Tensor:
    edges = []
    pose_edges = [
        (0,1),(0,4),(1,2),(2,3),(3,7),(4,5),(5,6),(6,8),(9,10),(11,12),
        (11,13),(13,15),(15,17),(15,19),(15,21),(12,14),(14,16),(16,18),(16,20),(16,22),
        (11,23),(12,24),(23,24),(23,25),(25,27),(27,29),(27,31),(24,26),(26,28),(28,30),(28,32),
    ]
    edges.extend(pose_edges)
    hand_base = [
        (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(0,9),(9,10),(10,11),(11,12),
        (0,13),(13,14),(14,15),(15,16),(0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17),
    ]
    for (a, b) in hand_base:
        edges.append((a + 33, b + 33))
    for (a, b) in hand_base:
        edges.append((a + 54, b + 54))
    edges.append((15, 33))  # left wrist -> left hand
    edges.append((16, 54))  # right wrist -> right hand
    A = torch.zeros(num_joints, num_joints, dtype=torch.float32)
    for (i, j) in edges:
        if i < num_joints and j < num_joints:
            A[i, j] = 1.0
            A[j, i] = 1.0
    A = A + torch.eye(num_joints, dtype=torch.float32)
    D = A.sum(dim=1)
    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D.clamp(min=1e-8)))
    return D_inv_sqrt @ A @ D_inv_sqrt


class GraphConvolution(nn.Module):
    def __init__(self, in_ch, out_ch, A, num_subsets=3):
        super().__init__()
        self.weights = nn.ParameterList([nn.Parameter(torch.empty(in_ch, out_ch)) for _ in range(num_subsets)])
        self.register_buffer('A', A)
        self.adaptive_A = nn.Parameter(torch.zeros_like(A))
        self.bn = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        for w in self.weights:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))

    def forward(self, x):
        A = F.softmax(self.A + self.adaptive_A, dim=-1)
        out = sum(torch.matmul(torch.matmul(A, x), w) for w in self.weights)
        return self.relu(self.bn(out.transpose(1, 2)).transpose(1, 2))


class SpatialGraphBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, dropout=0.1):
        super().__init__()
        self.gcn = GraphConvolution(in_ch, out_ch, A)
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Sequential(nn.Linear(in_ch, out_ch), nn.BatchNorm1d(out_ch)) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        if isinstance(self.residual, nn.Identity):
            res = x
        else:
            res = self.residual[0](x)
            res = self.residual[1](res.transpose(1, 2)).transpose(1, 2)
        return F.relu(self.dropout(self.gcn(x)) + res)


class TemporalConvBlock(nn.Module):
    def __init__(self, ch, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2
        self.conv = nn.Sequential(
            nn.Conv1d(ch, ch, kernel_size, padding=pad, dilation=dilation),
            nn.BatchNorm1d(ch), nn.ReLU(True), nn.Dropout(dropout),
            nn.Conv1d(ch, ch, kernel_size, padding=pad, dilation=dilation),
            nn.BatchNorm1d(ch),
        )

    def forward(self, x):
        return F.relu(x + self.conv(x))


class MultiScaleTemporalConv(nn.Module):
    def __init__(self, ch, num_scales=4, dropout=0.1):
        super().__init__()
        self.branches = nn.ModuleList([TemporalConvBlock(ch, dilation=2**i, dropout=dropout) for i in range(num_scales)])
        self.fusion = nn.Sequential(nn.Conv1d(ch * num_scales, ch, 1), nn.BatchNorm1d(ch), nn.ReLU(True))

    def forward(self, x):
        return self.fusion(torch.cat([b(x) for b in self.branches], dim=1))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, ff_dim=1024, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model), nn.Dropout(dropout)
        )

    def forward(self, x, mask=None):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, key_padding_mask=mask)
        x = x + h
        return x + self.ff(self.norm2(x))


class SignTranslatorNet(nn.Module):
    def __init__(self, in_ch=3, num_joints=75, hidden=256, num_classes=1926,
                 gcn_layers=4, gcn_drop=0.1, tf_heads=8, tf_layers=4,
                 tf_dim=256, tf_drop=0.1, tf_ff=1024, num_scales=4, max_seq=150):
        super().__init__()
        self.num_joints = num_joints
        self.hidden_dim = hidden
        A = build_skeleton_adjacency(num_joints)
        self.register_buffer('A', A)

        dims = [in_ch] + [hidden // 2] * (gcn_layers - 1) + [hidden]
        self.spatial_gcn = nn.ModuleList([SpatialGraphBlock(dims[i], dims[i+1], A, gcn_drop) for i in range(gcn_layers)])
        self.joint_pool = nn.Sequential(
            nn.Linear(num_joints * hidden, hidden), nn.LayerNorm(hidden),
            nn.ReLU(True), nn.Dropout(gcn_drop)
        )
        self.temporal_conv = MultiScaleTemporalConv(hidden, num_scales, gcn_drop)
        self.pos_encoding = PositionalEncoding(tf_dim, max_seq * 2, tf_drop)
        self.transformer_layers = nn.ModuleList([TransformerBlock(tf_dim, tf_heads, tf_ff, tf_drop) for _ in range(tf_layers)])
        self.transformer_norm = nn.LayerNorm(tf_dim)
        self.classifier = nn.Sequential(
            nn.Linear(tf_dim, tf_dim), nn.GELU(), nn.Dropout(tf_drop),
            nn.Linear(tf_dim, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, mask=None):
        B, T, J, C = x.shape
        x = x.reshape(B * T, J, C)
        for gcn in self.spatial_gcn:
            x = gcn(x)
        x = self.joint_pool(x.reshape(B * T, J * self.hidden_dim)).reshape(B, T, self.hidden_dim)
        x = self.temporal_conv(x.transpose(1, 2)).transpose(1, 2)
        x = self.pos_encoding(x)
        for layer in self.transformer_layers:
            x = layer(x, mask=mask)
        x = self.transformer_norm(x)
        if mask is not None:
            m = (~mask).unsqueeze(-1).float()
            x = (x * m).sum(1) / m.sum(1).clamp(min=1)
        else:
            x = x.mean(1)
        return self.classifier(x)


# Verificare rapida
model_test = SignTranslatorNet(num_classes=100)
print(f"Parametri: {sum(p.numel() for p in model_test.parameters()):,}")
dummy = torch.randn(2, 30, 75, 3)
with torch.no_grad():
    out = model_test(dummy)
print(f"Input: {dummy.shape} -> Output: {out.shape}")
del model_test
print("Model OK!")

Parametri: 10,264,776
Input: torch.Size([2, 30, 75, 3]) -> Output: torch.Size([2, 100])
Model OK!


## 3. Dataset + Augmentari

In [14]:
import io
import pickle
import random
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

NUM_JOINTS = 75
MAX_SEQ_LEN = 150


class SignAugmentation:
    def __init__(self, training=True):
        self.training = training

    def __call__(self, joints, vis):
        if not self.training:
            return joints, vis
        # Flip orizontal
        if random.random() < 0.5:
            joints = joints.copy()
            joints[:, :, 0] = -joints[:, :, 0]
            swap = [(1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
                    (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32)]
            for a, b in swap:
                joints[:, [a, b]] = joints[:, [b, a]]
            lh = joints[:, 33:54].copy()
            joints[:, 33:54] = joints[:, 54:75]
            joints[:, 54:75] = lh
        # Scalare
        if random.random() < 0.7:
            joints[:, :, :2] *= random.uniform(0.8, 1.2)
        # Rotatie
        if random.random() < 0.5:
            rad = np.deg2rad(random.uniform(-15, 15))
            c, s = np.cos(rad), np.sin(rad)
            cx, cy = np.nanmean(joints[:,:,0]), np.nanmean(joints[:,:,1])
            x, y = joints[:,:,0]-cx, joints[:,:,1]-cy
            joints = joints.copy()
            joints[:,:,0] = x*c - y*s + cx
            joints[:,:,1] = x*s + y*c + cy
        # Zgomot
        if random.random() < 0.5:
            joints = joints + np.random.randn(*joints.shape).astype(np.float32) * 0.01
        # Temporal drop (INAINTE de padding)
        if random.random() < 0.3 and joints.shape[0] > 5:
            keep = sorted(random.sample(range(joints.shape[0]), max(2, int(joints.shape[0]*0.9))))
            joints, vis = joints[keep], vis[keep]
        return joints, vis


class CpuUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu', weights_only=False)
        return super().find_class(module, name)


def pad_or_crop(joints, vis, max_len, num_j):
    T, J_orig, C = joints.shape
    if J_orig >= num_j:
        joints = joints[:, :num_j]
    else:
        joints = np.concatenate([joints, np.zeros((T, num_j - J_orig, C), dtype=np.float32)], axis=1)
    v = np.ones((T, num_j), dtype=np.float32)
    if J_orig < num_j:
        v[:, J_orig:] = 0
    if T > max_len:
        idx = np.linspace(0, T-1, max_len, dtype=int)
        joints, v = joints[idx], v[idx]
    elif T < max_len:
        joints = np.concatenate([joints, np.zeros((max_len-T, num_j, C), dtype=np.float32)])
        v = np.concatenate([v, np.zeros((max_len-T, num_j), dtype=np.float32)])
    return joints, v


def normalize(joints, vis):
    valid = vis > 0.5
    if valid.any():
        for t in range(joints.shape[0]):
            if valid[t].any():
                joints[t] -= joints[t, valid[t]].mean(0)
        vj = joints[valid.any(axis=1)]
        if len(vj) > 0:
            mx = np.abs(vj[:, :, :2]).max()
            if mx > 1e-6:
                joints[:, :, :2] /= mx
    return joints


class How2SignDataset(Dataset):
    def __init__(self, pkl_dir, max_pkls=None, augment=True):
        self.pkl_dir = pkl_dir
        self.aug = SignAugmentation(augment)
        self.files = sorted([f for f in os.listdir(pkl_dir) if f.endswith('.pkl')])
        if max_pkls:
            self.files = self.files[:max_pkls]
        print(f"How2Sign: {len(self.files)} secvente")

    def __len__(self):
        return len(self.files)

    def _load(self, path):
        try:
            return torch.load(path, map_location='cpu', weights_only=False)
        except Exception:
            with open(path, 'rb') as f:
                return CpuUnpickler(f).load()

    def _extract(self, data):
        if isinstance(data, dict):
            for k in ['joints3d','joints','body_joints','keypoints3d','pred_joints']:
                if k in data:
                    v = data[k]
                    if isinstance(v, torch.Tensor): v = v.cpu().numpy()
                    if isinstance(v, np.ndarray) and v.ndim == 3 and v.shape[-1] == 3:
                        return v.astype(np.float32)
            for v in data.values():
                if isinstance(v, dict):
                    r = self._extract(v)
                    if r is not None: return r
                elif isinstance(v, (torch.Tensor, np.ndarray)):
                    if isinstance(v, torch.Tensor): v = v.cpu().numpy()
                    if v.ndim == 3 and v.shape[-1] == 3 and v.shape[1] > 10:
                        return v.astype(np.float32)
        elif isinstance(data, (torch.Tensor, np.ndarray)):
            if isinstance(data, torch.Tensor): data = data.cpu().numpy()
            if data.ndim == 3 and data.shape[-1] == 3:
                return data.astype(np.float32)
        return None

    def __getitem__(self, idx):
        try:
            joints = self._extract(self._load(os.path.join(self.pkl_dir, self.files[idx])))
        except Exception:
            joints = None
        if joints is None:
            return {'joints': torch.zeros(MAX_SEQ_LEN, NUM_JOINTS, 3),
                    'vis': torch.zeros(MAX_SEQ_LEN, NUM_JOINTS),
                    'mask': torch.ones(MAX_SEQ_LEN, dtype=torch.bool),
                    'label': torch.tensor(-1)}
        T_orig = joints.shape[0]
        vis_tmp = np.ones((joints.shape[0], joints.shape[1]), dtype=np.float32)
        # Augmentare INAINTE de padding
        joints, vis_tmp = self.aug(joints, vis_tmp)
        T_orig = min(T_orig, joints.shape[0])
        joints, vis = pad_or_crop(joints, vis_tmp, MAX_SEQ_LEN, NUM_JOINTS)
        joints = normalize(joints, vis)
        mask = torch.zeros(MAX_SEQ_LEN, dtype=torch.bool)
        if T_orig < MAX_SEQ_LEN:
            mask[T_orig:] = True
        return {'joints': torch.from_numpy(joints).float(),
                'vis': torch.from_numpy(vis).float(),
                'mask': mask, 'label': torch.tensor(-1)}


class RomanianSignDataset(Dataset):
    def __init__(self, dataset_json, videos_dir, split='train', augment=True, cache_dir=None):
        self.videos_dir = videos_dir
        self.aug = SignAugmentation(augment and split == 'train')
        self.cache_dir = cache_dir
        if cache_dir:
            os.makedirs(cache_dir, exist_ok=True)
        with open(dataset_json, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.glosses = sorted(set(e['gloss'] for e in data))
        self.gloss_to_idx = {g: i for i, g in enumerate(self.glosses)}
        self.num_classes = len(self.glosses)
        self.samples = []
        for e in data:
            label = self.gloss_to_idx[e['gloss']]
            for inst in e['instances']:
                if inst['split'] == split:
                    vp = os.path.join(videos_dir, f"{inst['video_id']}.mp4")
                    if os.path.exists(vp):
                        self.samples.append({'path': vp, 'id': inst['video_id'], 'label': label})
        print(f"RO [{split}]: {len(self.samples)} instante, {self.num_classes} clase")

    def __len__(self):
        return len(self.samples)

    def _extract_mp(self, video_path):
        import mediapipe as mp
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return np.zeros((1, NUM_JOINTS, 3), dtype=np.float32), np.zeros((1, NUM_JOINTS), dtype=np.float32)
        all_j, all_v = [], []
        with mp.solutions.holistic.Holistic(static_image_mode=False, model_complexity=2, min_detection_confidence=0.5) as h:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                res = h.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                j = np.zeros((NUM_JOINTS, 3), dtype=np.float32)
                v = np.zeros(NUM_JOINTS, dtype=np.float32)
                if res.pose_landmarks:
                    for i, lm in enumerate(res.pose_landmarks.landmark):
                        if i < 33:
                            j[i] = [lm.x, lm.y, lm.z]
                            v[i] = 1.0 if lm.visibility > 0.5 else 0.0
                if res.left_hand_landmarks:
                    for i, lm in enumerate(res.left_hand_landmarks.landmark):
                        if i < 21:
                            j[33+i] = [lm.x, lm.y, lm.z]
                            v[33+i] = 1.0
                if res.right_hand_landmarks:
                    for i, lm in enumerate(res.right_hand_landmarks.landmark):
                        if i < 21:
                            j[54+i] = [lm.x, lm.y, lm.z]
                            v[54+i] = 1.0
                all_j.append(j)
                all_v.append(v)
        cap.release()
        if not all_j:
            return np.zeros((1, NUM_JOINTS, 3), dtype=np.float32), np.zeros((1, NUM_JOINTS), dtype=np.float32)
        return np.array(all_j, dtype=np.float32), np.array(all_v, dtype=np.float32)

    def _get(self, path, vid):
        if self.cache_dir:
            cp = os.path.join(self.cache_dir, f"{vid}.npz")
            if os.path.exists(cp):
                d = np.load(cp)
                return d['joints'], d['vis']
        j, v = self._extract_mp(path)
        if self.cache_dir:
            np.savez_compressed(os.path.join(self.cache_dir, f"{vid}.npz"), joints=j, vis=v)
        return j, v

    def __getitem__(self, idx):
        s = self.samples[idx]
        joints, vis = self._get(s['path'], s['id'])
        T_orig = joints.shape[0]
        joints, vis = self.aug(joints, vis)
        T_orig = min(T_orig, joints.shape[0])
        joints, vis = pad_or_crop(joints, vis, MAX_SEQ_LEN, NUM_JOINTS)
        joints = normalize(joints, vis)
        mask = torch.zeros(MAX_SEQ_LEN, dtype=torch.bool)
        if T_orig < MAX_SEQ_LEN:
            mask[T_orig:] = True
        return {'joints': torch.from_numpy(joints).float(),
                'vis': torch.from_numpy(vis).float(),
                'mask': mask,
                'label': torch.tensor(s['label'], dtype=torch.long)}


print("Dataset + Augmentari OK!")

Dataset + Augmentari OK!


## 4. Pre-antrenare Self-Supervised pe How2Sign

2 task-uri:
- **Masked Joint Prediction (MJP)** — ascunde 15% din articulatii, prezice-le
- **Temporal Order Prediction (TOP)** — detecteaza daca frame-urile sunt amestecate

In [ ]:
# === COPIERE DATE PE DISC LOCAL (doar pe Colab) ===
if IS_COLAB:
    import shutil
    from tqdm.auto import tqdm as tqdm_copy
    LOCAL_PKL_DIR = '/content/how2sign_pkls'

    n_drive = len([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
    n_local = len([f for f in os.listdir(LOCAL_PKL_DIR) if f.endswith('.pkl')]) if os.path.isdir(LOCAL_PKL_DIR) else 0

    if n_local < n_drive * 0.95:
        print(f"Pe Drive: {n_drive} pkl-uri | Pe disc local: {n_local}")
        if os.path.exists(LOCAL_PKL_DIR):
            shutil.rmtree(LOCAL_PKL_DIR)
        os.makedirs(LOCAL_PKL_DIR, exist_ok=True)
        src_files = sorted([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
        for f in tqdm_copy(src_files, desc='Copiere', unit='fisier'):
            shutil.copy2(os.path.join(HOW2SIGN_PKL_DIR, f), os.path.join(LOCAL_PKL_DIR, f))
        n_local = len(os.listdir(LOCAL_PKL_DIR))
        print(f"\nGata! {n_local} fisiere copiate.")
    else:
        print(f"Pkl-uri deja pe disc local: {n_local}/{n_drive}")

    HOW2SIGN_PKL_DIR = LOCAL_PKL_DIR
    print(f"Se foloseste: {HOW2SIGN_PKL_DIR}")
else:
    print("Rulare locala — nu e nevoie de copiere.")

Pe Drive: 30997 pkl-uri | Pe disc local: 120


Copiere:   0%|          | 0/30997 [00:00<?, ?fisier/s]

In [ ]:
import os
n = len([f for f in os.listdir('/content/how2sign_pkls') if f.endswith('.pkl')]) if os.path.isdir('/content/how2sign_pkls') else 0
print(f"Fisiere copiate pana acum: {n}/30997")


In [ ]:
# === COPIERE DATE PE DISC LOCAL (doar pe Colab) ===
if IS_COLAB:
    import shutil
    LOCAL_PKL_DIR = '/content/how2sign_pkls'

    n_drive = len([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
    n_local = len([f for f in os.listdir(LOCAL_PKL_DIR) if f.endswith('.pkl')]) if os.path.isdir(LOCAL_PKL_DIR) else 0

    if n_local < n_drive * 0.95:
        print(f"Pe Drive: {n_drive} pkl-uri | Pe disc local: {n_local}")
        if os.path.exists(LOCAL_PKL_DIR):
            shutil.rmtree(LOCAL_PKL_DIR)
        os.makedirs(LOCAL_PKL_DIR, exist_ok=True)
        src_files = sorted([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
        for f in tqdm(src_files, desc='Copiere pkl-uri'):
            shutil.copy2(os.path.join(HOW2SIGN_PKL_DIR, f), os.path.join(LOCAL_PKL_DIR, f))
        n_local = len(os.listdir(LOCAL_PKL_DIR))
        print(f"Gata! {n_local} fisiere copiate.")
    else:
        print(f"Pkl-uri deja pe disc local: {n_local}/{n_drive}")

    HOW2SIGN_PKL_DIR = LOCAL_PKL_DIR
    print(f"Se foloseste: {HOW2SIGN_PKL_DIR}")
else:
    print("Rulare locala — nu e nevoie de copiere.")

: 

In [ ]:
# === COPIERE DATE PE DISC LOCAL (doar pe Colab) ===
if IS_COLAB:
    import shutil
    LOCAL_PKL_DIR = '/content/how2sign_pkls'

    n_drive = len([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
    n_local = len([f for f in os.listdir(LOCAL_PKL_DIR) if f.endswith('.pkl')]) if os.path.isdir(LOCAL_PKL_DIR) else 0

    if n_local < n_drive * 0.95:
        print(f"Pe Drive: {n_drive} pkl-uri | Pe disc local: {n_local}")
        if os.path.exists(LOCAL_PKL_DIR):
            shutil.rmtree(LOCAL_PKL_DIR)
        os.makedirs(LOCAL_PKL_DIR, exist_ok=True)
        src_files = sorted([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
        for f in tqdm(src_files, desc='Copiere pkl-uri'):
            shutil.copy2(os.path.join(HOW2SIGN_PKL_DIR, f), os.path.join(LOCAL_PKL_DIR, f))
        n_local = len(os.listdir(LOCAL_PKL_DIR))
        print(f"Gata! {n_local} fisiere copiate.")
    else:
        print(f"Pkl-uri deja pe disc local: {n_local}/{n_drive}")

    HOW2SIGN_PKL_DIR = LOCAL_PKL_DIR
    print(f"Se foloseste: {HOW2SIGN_PKL_DIR}")
else:
    print("Rulare locala — nu e nevoie de copiere.")

Pe Drive: 30997 pkl-uri | Pe disc local: 1220


Copiere pkl-uri:   0%|          | 0/30997 [00:00<?, ?it/s]

In [ ]:
# === COPIERE DATE PE DISC LOCAL (doar pe Colab) ===
if IS_COLAB:
    import shutil
    LOCAL_PKL_DIR = '/content/how2sign_pkls'

    n_drive = len([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
    n_local = len([f for f in os.listdir(LOCAL_PKL_DIR) if f.endswith('.pkl')]) if os.path.isdir(LOCAL_PKL_DIR) else 0

    if n_local < n_drive * 0.95:
        print(f"Pe Drive: {n_drive} pkl-uri | Pe disc local: {n_local}")
        if os.path.exists(LOCAL_PKL_DIR):
            shutil.rmtree(LOCAL_PKL_DIR)
        os.makedirs(LOCAL_PKL_DIR, exist_ok=True)
        src_files = sorted([f for f in os.listdir(HOW2SIGN_PKL_DIR) if f.endswith('.pkl')])
        for f in tqdm(src_files, desc='Copiere pkl-uri'):
            shutil.copy2(os.path.join(HOW2SIGN_PKL_DIR, f), os.path.join(LOCAL_PKL_DIR, f))
        n_local = len(os.listdir(LOCAL_PKL_DIR))
        print(f"Gata! {n_local} fisiere copiate.")
    else:
        print(f"Pkl-uri deja pe disc local: {n_local}/{n_drive}")

    HOW2SIGN_PKL_DIR = LOCAL_PKL_DIR
    print(f"Se foloseste: {HOW2SIGN_PKL_DIR}")
else:
    print("Rulare locala — nu e nevoie de copiere.")

Pe Drive: 30997 pkl-uri | Pe disc local: 1220


Copiere pkl-uri:   0%|          | 0/30997 [00:00<?, ?it/s]

In [ ]:
# === PRE-ANTRENARE ===
PRETRAIN_EPOCHS = 50
PRETRAIN_BS = 32
PRETRAIN_LR = 1e-3
HIDDEN = 256
NUM_WORKERS = 2 if IS_COLAB else 4
PRETRAIN_PATH = os.path.join(MODELS_DIR, 'pretrained_best.pth')

dataset = How2SignDataset(HOW2SIGN_PKL_DIR)
N = len(dataset)
val_n = max(1, int(N * 0.05))
train_set, val_set = random_split(dataset, [N - val_n, val_n], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=PRETRAIN_BS, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=PRETRAIN_BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PretrainModel(hidden=HIDDEN).to(device)
print(f"Device: {device}")
print(f"Parametri: {sum(p.numel() for p in model.parameters()):,}")
print(f"Workers: {NUM_WORKERS}")

optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=1e-4)
total_steps = PRETRAIN_EPOCHS * len(train_loader)
warmup_steps = 5 * len(train_loader)

def lr_fn(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(1e-7 / PRETRAIN_LR, 0.5 * (1 + math.cos(math.pi * p)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)
best_val = float('inf')

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    model.train()
    ep_loss, ep_mjp, ep_top = 0, 0, 0
    pbar = tqdm(train_loader, desc=f'Pre-train {epoch}/{PRETRAIN_EPOCHS}')
    for batch in pbar:
        j = batch['joints'].to(device)
        v = batch['vis'].to(device)
        m = batch['mask'].to(device)
        mp, tp, jo, mm, tl = model(j, v, m)
        loss, ml, tl_v = pretrain_loss(mp, tp, jo, mm, tl)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        ep_loss += loss.item()
        ep_mjp += ml
        ep_top += tl_v
        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')
    n = len(train_loader)
    ep_loss /= n
    ep_mjp /= n
    ep_top /= n

    # Validare
    model.eval()
    vl = 0
    with torch.no_grad():
        for batch in val_loader:
            j = batch['joints'].to(device)
            v = batch['vis'].to(device)
            m = batch['mask'].to(device)
            mp, tp, jo, mm, tl = model(j, v, m)
            loss, _, _ = pretrain_loss(mp, tp, jo, mm, tl)
            vl += loss.item()
    vl /= max(len(val_loader), 1)

    mk = ''
    if vl < best_val:
        best_val = vl
        torch.save({'backbone': model.backbone.state_dict(), 'epoch': epoch, 'val_loss': best_val}, PRETRAIN_PATH)
        mk = ' << BEST'
    print(f'Ep {epoch:3d} | Loss: {ep_loss:.4f} (MJP:{ep_mjp:.4f} TOP:{ep_top:.4f}) | Val: {vl:.4f}{mk}')

print(f'\nPre-antrenare gata! Best val loss: {best_val:.4f}')

How2Sign: 1576 secvente
Device: cuda
Parametri: 10,330,567
Workers: 2


Pre-train 1/50:   0%|          | 0/46 [00:00<?, ?it/s]

Ep   1 | Loss: nan (MJP:nan TOP:nan) | Val: nan


Pre-train 2/50:   0%|          | 0/46 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e0252fdcb80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e0252fdcb80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Ep   2 | Loss: nan (MJP:nan TOP:nan) | Val: nan


Pre-train 3/50:   0%|          | 0/46 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e0252fdcb80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e0252fdcb80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

KeyboardInterrupt: 

: 

## 5. Cache keypoints din videourile romanesti

Ruleaza o singura data — extrage keypoints cu MediaPipe si le salveaza local.

In [ ]:
import mediapipe as mp

all_vids = set()
for e in ro_data:
    for inst in e['instances']:
        vp = os.path.join(RO_VIDEOS_DIR, f"{inst['video_id']}.mp4")
        if os.path.exists(vp):
            all_vids.add((inst['video_id'], vp))

print(f"Total videouri de procesat: {len(all_vids)}")
processed, skipped = 0, 0

for vid, vpath in tqdm(all_vids, desc='Cache keypoints'):
    cp = os.path.join(CACHE_DIR, f"{vid}.npz")
    if os.path.exists(cp):
        skipped += 1
        continue
    try:
        cap = cv2.VideoCapture(vpath)
        if not cap.isOpened():
            skipped += 1
            continue
        aj, av = [], []
        with mp.solutions.holistic.Holistic(static_image_mode=False, model_complexity=2, min_detection_confidence=0.5) as h:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                res = h.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                j = np.zeros((NUM_JOINTS, 3), dtype=np.float32)
                v = np.zeros(NUM_JOINTS, dtype=np.float32)
                if res.pose_landmarks:
                    for i, lm in enumerate(res.pose_landmarks.landmark):
                        if i < 33:
                            j[i] = [lm.x, lm.y, lm.z]
                            v[i] = 1.0 if lm.visibility > 0.5 else 0.0
                if res.left_hand_landmarks:
                    for i, lm in enumerate(res.left_hand_landmarks.landmark):
                        if i < 21:
                            j[33+i] = [lm.x, lm.y, lm.z]
                            v[33+i] = 1.0
                if res.right_hand_landmarks:
                    for i, lm in enumerate(res.right_hand_landmarks.landmark):
                        if i < 21:
                            j[54+i] = [lm.x, lm.y, lm.z]
                            v[54+i] = 1.0
                aj.append(j)
                av.append(v)
        cap.release()
        if aj:
            np.savez_compressed(cp, joints=np.array(aj, dtype=np.float32), vis=np.array(av, dtype=np.float32))
            processed += 1
        else:
            skipped += 1
    except Exception as e:
        print(f"Eroare {vid}: {e}")
        skipped += 1

print(f"\nProcesate: {processed} | Sarite/cache: {skipped}")

## 6. Fine-tuning pe date romanesti

In [ ]:
# === FINE-TUNING ===
FT_EPOCHS = 80
FT_BS = 16
FT_LR = 1e-4
FT_PATH = os.path.join(MODELS_DIR, 'finetuned_best.pth')

train_ds = RomanianSignDataset(RO_DATASET_JSON, RO_VIDEOS_DIR, split='train', augment=True, cache_dir=CACHE_DIR)
test_ds = RomanianSignDataset(RO_DATASET_JSON, RO_VIDEOS_DIR, split='test', augment=False, cache_dir=CACHE_DIR)
NUM_CLASSES = train_ds.num_classes

N = len(train_ds)
vn = max(1, int(N * 0.1))
tr_set, vl_set = random_split(train_ds, [N - vn, vn], generator=torch.Generator().manual_seed(42))
tr_loader = DataLoader(tr_set, batch_size=FT_BS, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
vl_loader = DataLoader(vl_set, batch_size=FT_BS, shuffle=False, num_workers=4, pin_memory=True)
te_loader = DataLoader(test_ds, batch_size=FT_BS, shuffle=False, num_workers=4, pin_memory=True)

# Model cu numarul corect de clase
model = SignTranslatorNet(num_classes=NUM_CLASSES, hidden=HIDDEN).to(device)

# Incarca backbone pre-antrenat
ckpt = torch.load(PRETRAIN_PATH, map_location=device, weights_only=False)
sd = ckpt['backbone']
md = model.state_dict()
loaded = {k: v for k, v in sd.items() if k in md and md[k].shape == v.shape}
md.update(loaded)
model.load_state_dict(md, strict=False)
print(f"Incarcat {len(loaded)}/{len(md)} parametri pre-antrenati")
print(f"Clase: {NUM_CLASSES} | Parametri: {sum(p.numel() for p in model.parameters()):,}")

# Salvare gloss mapping
gloss_path = os.path.join(MODELS_DIR, 'gloss_mapping.json')
with open(gloss_path, 'w', encoding='utf-8') as f:
    json.dump({'glosses': train_ds.glosses, 'gloss_to_idx': train_ds.gloss_to_idx, 'num_classes': NUM_CLASSES}, f, ensure_ascii=False, indent=2)
print(f"Gloss mapping salvat: {gloss_path}")

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# LR diferentiat: backbone mic, classifier mare
bb_p = [p for n, p in model.named_parameters() if 'classifier' not in n]
cl_p = [p for n, p in model.named_parameters() if 'classifier' in n]
optimizer = torch.optim.AdamW([
    {'params': bb_p, 'lr': FT_LR},
    {'params': cl_p, 'lr': FT_LR * 10}
], weight_decay=1e-4)

total_steps = FT_EPOCHS * len(tr_loader)
warmup_steps = 3 * len(tr_loader)

def lr_fn(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(1e-7 / FT_LR, 0.5 * (1 + math.cos(math.pi * p)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)
best_acc = 0.0
patience = 0

for epoch in range(1, FT_EPOCHS + 1):
    model.train()
    ep_loss, correct, total = 0, 0, 0
    pbar = tqdm(tr_loader, desc=f'FT {epoch}/{FT_EPOCHS}')
    for batch in pbar:
        j = batch['joints'].to(device)
        m = batch['mask'].to(device)
        lab = batch['label'].to(device)
        # Mixup
        lam = max(np.random.beta(0.2, 0.2), 0.5) if random.random() < 0.5 else 1.0
        if lam < 1.0:
            idx = torch.randperm(j.size(0), device=device)
            j_mix = lam * j + (1 - lam) * j[idx]
            loss = lam * criterion(model(j_mix, m), lab) + (1 - lam) * criterion(model(j_mix, m), lab[idx])
        else:
            loss = criterion(model(j, m), lab)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        ep_loss += loss.item()
        if lam == 1.0:
            with torch.no_grad():
                correct += model(j, m).argmax(1).eq(lab).sum().item()
        total += lab.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    # Validare
    model.eval()
    vc, vt, v5 = 0, 0, 0
    with torch.no_grad():
        for batch in vl_loader:
            j = batch['joints'].to(device)
            m = batch['mask'].to(device)
            lab = batch['label'].to(device)
            logits = model(j, m)
            vc += logits.argmax(1).eq(lab).sum().item()
            vt += lab.size(0)
            _, t5 = logits.topk(min(5, logits.size(1)), dim=1)
            v5 += sum(lab[i] in t5[i] for i in range(lab.size(0)))
    va = 100.0 * vc / max(vt, 1)
    v5a = 100.0 * v5 / max(vt, 1)

    mk = ''
    if va > best_acc:
        best_acc = va
        patience = 0
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_acc': best_acc, 'num_classes': NUM_CLASSES}, FT_PATH)
        mk = ' << BEST'
    else:
        patience += 1
    print(f'Ep {epoch:3d} | Loss: {ep_loss/len(tr_loader):.4f} | Val Acc: {va:.1f}% Top5: {v5a:.1f}%{mk}')
    if patience >= 15:
        print(f'Early stopping la epoca {epoch}')
        break

# Test final
print('\n' + '='*50)
print('TEST FINAL')
print('='*50)
ckpt = torch.load(FT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
tc, tt, t5 = 0, 0, 0
with torch.no_grad():
    for batch in tqdm(te_loader, desc='Test'):
        j = batch['joints'].to(device)
        m = batch['mask'].to(device)
        lab = batch['label'].to(device)
        logits = model(j, m)
        tc += logits.argmax(1).eq(lab).sum().item()
        tt += lab.size(0)
        _, tp = logits.topk(min(5, logits.size(1)), dim=1)
        t5 += sum(lab[i] in tp[i] for i in range(lab.size(0)))
print(f'Test Accuracy: {100*tc/max(tt,1):.2f}%')
print(f'Test Top-5:    {100*t5/max(tt,1):.2f}%')
print(f'Best Val Acc:  {best_acc:.2f}%')

## 7. Verificare + Export model final

In [ ]:
# Verificare ca modelul se incarca corect
ckpt = torch.load(FT_PATH, map_location='cpu', weights_only=False)
m = SignTranslatorNet(num_classes=ckpt['num_classes'])
m.load_state_dict(ckpt['model'])
m.eval()
with torch.no_grad():
    out = m(torch.randn(1, 30, 75, 3))
print(f"Output: {out.shape} ({ckpt['num_classes']} clase)")
print(f"Val Acc: {ckpt['val_acc']:.2f}%")
print(f"\nModelul functioneaza!")
print(f"Fisiere salvate in: {MODELS_DIR}")
print(f"  - pretrained_best.pth  (backbone pre-antrenat)")
print(f"  - finetuned_best.pth   (model final)")
print(f"  - gloss_mapping.json   (mapare glosuri -> indici)")

# Daca suntem pe Colab, copiaza modelele pe Drive
if IS_COLAB:
    import shutil
    DRIVE_MODELS = f'{DRIVE_BASE}/models'
    os.makedirs(DRIVE_MODELS, exist_ok=True)
    for src in [PRETRAIN_PATH, FT_PATH, os.path.join(MODELS_DIR, 'gloss_mapping.json')]:
        if os.path.exists(src):
            dst = os.path.join(DRIVE_MODELS, os.path.basename(src))
            shutil.copy2(src, dst)
            print(f"Copiat pe Drive: {os.path.basename(src)}")
    print(f"\nModele pe Drive: {DRIVE_MODELS}")
    print("Descarca finetuned_best.pth + gloss_mapping.json pe PC in SignTranslator/models/")